# Cleaning economic data

Data is obtained [here](https://data.cccnewyork.org/data/map/66/median-incomes#66/39/3/107/131/a/a), but it seems like they've compiled data from other open data sources like the census and NY Open data



## Import and clean median income

In [4]:
import pandas as pd

In [5]:
# Median income from census, taking only 2023 and all household data

median_df = pd.read_csv('raw_data/Median Incomes.csv')
median_df

,Location,Household Type,TimeFrame,DataFormat,Data,Fips
0,Astoria,All Households,2023,Dollars,84590,401
1,Astoria,Families,2023,Dollars,94918,401
2,Astoria,Families with Children,2023,Dollars,85568,401
3,Astoria,Families without Children,2023,Dollars,110222,401
4,Battery Park/Tribeca,All Households,2023,Dollars,198945,101
...,...,...,...,...,...,...
4675,Hunts Point,Families,2005,Dollars,25298.11273,202
4676,Mott Haven,All Households,2005,Dollars,21038.11098,201
4677,Hunts Point,All Households,2005,Dollars,21038.11098,202
4678,Mott Haven,Families with Children,2005,Dollars,18714.33604,201


In [6]:
median_df = median_df[(median_df['TimeFrame'] == 2023) & 
          (median_df['Household Type'] == "All Households")]

median_df.rename(columns = {"Fips" : "CD",
                            "Data" : "Med_Income"}, inplace=True)

C:\Users\Jason\AppData\Local\Temp\ipykernel_24372\3627457581.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  median_df.rename(columns = {"Fips" : "CD",


In [7]:
median_df.drop(columns = ['Household Type'], inplace=True)

C:\Users\Jason\AppData\Local\Temp\ipykernel_24372\2704775050.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  median_df.drop(columns = ['Household Type'], inplace=True)


## Calculate Z-scores

In [14]:
import geopandas as gpd
import numpy as np

In [11]:
gdf = gpd.read_file('cleaned_data/facilities_w_districts/facilities_w_districts.shp')
gdf.head(5)

,DISTRICTCO,CITY OPERA,EDUCATION,HEALTH AND,OTHER,PARKS AND,PUBLIC SAF,TRANSPORTA,total_faci,CD Name,Shape_Leng,Shape_Area,city,BoroCD,geometry
0,101,143,34,5,0,34,18,13,124,"Battery Park City, Tribeca",69090.913150,4.169373e+07,NEW YORK,101,"MULTIPOLYGON (((972081.788 190733.467, 972184...."
1,102,9,20,2,0,40,8,4,80,"Greenwich Village, Soho",35009.716634,3.772401e+07,NEW YORK,102,"POLYGON ((981713.541 209788.14, 981750.999 209..."
2,103,27,68,35,1,87,11,6,194,"Lower East Side, Chinatown",30505.700417,4.687884e+07,NEW YORK,103,"POLYGON ((987359.695 206680.171, 987694.33 206..."
3,104,19,47,4,0,18,10,10,99,"Chelsea, Clinton",67546.045387,4.931005e+07,NEW YORK,104,"POLYGON ((985929.324 220967.002, 985995.954 22..."
4,105,12,29,7,0,20,7,4,71,Midtown Business District,35288.334291,4.379004e+07,NEW YORK,105,"POLYGON ((991725.244 217725.299, 992169.505 21..."


In [ ]:
avg = gdf['total_faci'].aggregate('mean')
stdev = np.std(gdf['total_faci'])

gdf['z_score'] = (gdf['total_faci'] - avg) / stdev
gdf.head(5)

,DISTRICTCO,CITY OPERA,EDUCATION,HEALTH AND,OTHER,PARKS AND,PUBLIC SAF,TRANSPORTA,total_faci,CD Name,Shape_Leng,Shape_Area,city,BoroCD,geometry,z_score
0,101,143,34,5,0,34,18,13,124,"Battery Park City, Tribeca",69090.913150,4.169373e+07,NEW YORK,101,"MULTIPOLYGON (((972081.788 190733.467, 972184....",0.326696
1,102,9,20,2,0,40,8,4,80,"Greenwich Village, Soho",35009.716634,3.772401e+07,NEW YORK,102,"POLYGON ((981713.541 209788.14, 981750.999 209...",-0.658325
2,103,27,68,35,1,87,11,6,194,"Lower East Side, Chinatown",30505.700417,4.687884e+07,NEW YORK,103,"POLYGON ((987359.695 206680.171, 987694.33 206...",1.893776
3,104,19,47,4,0,18,10,10,99,"Chelsea, Clinton",67546.045387,4.931005e+07,NEW YORK,104,"POLYGON ((985929.324 220967.002, 985995.954 22...",-0.232975
4,105,12,29,7,0,20,7,4,71,Midtown Business District,35288.334291,4.379004e+07,NEW YORK,105,"POLYGON ((991725.244 217725.299, 992169.505 21...",-0.859807


In [ ]:
gdf.to_file("facilities_w_districts.shp")

In [ ]:
median_df.drop(columns = ['DataFormat'], inplace=True)


,Location,TimeFrame,DataFormat,Med_Income,CD
0,Astoria,2023,Dollars,84590,401
4,Battery Park/Tribeca,2023,Dollars,198945,101
8,Bay Ridge,2023,Dollars,88566,310
12,Bayside,2023,Dollars,107607,411
16,Bedford Park,2023,Dollars,42387,207
...,...,...,...,...,...
240,Upper West Side,2023,Dollars,150017,107
244,Washington Heights,2023,Dollars,61527,112
248,Williamsbridge,2023,Dollars,67973,212
252,Williamsburg/Greenpoint,2023,Dollars,111492,301


In [25]:
median_df['Med Income Str'] = median_df['Med_Income'].astype(int).apply(
    lambda x: f"${x:,.0f}"
)
median_df

C:\Users\Jason\AppData\Local\Temp\ipykernel_24372\3388627990.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  median_df['Med Income Str'] = median_df['Med_Income'].astype(int).apply(


,Location,TimeFrame,DataFormat,Med_Income,CD,Med Income Str
0,Astoria,2023,Dollars,84590,401,"$84,590"
4,Battery Park/Tribeca,2023,Dollars,198945,101,"$198,945"
8,Bay Ridge,2023,Dollars,88566,310,"$88,566"
12,Bayside,2023,Dollars,107607,411,"$107,607"
16,Bedford Park,2023,Dollars,42387,207,"$42,387"
...,...,...,...,...,...,...
240,Upper West Side,2023,Dollars,150017,107,"$150,017"
244,Washington Heights,2023,Dollars,61527,112,"$61,527"
248,Williamsbridge,2023,Dollars,67973,212,"$67,973"
252,Williamsburg/Greenpoint,2023,Dollars,111492,301,"$111,492"


In [26]:
avg = median_df['Med_Income'].astype(int).aggregate('mean')
stdev = np.std(median_df['Med_Income'].astype(int))

median_df['z_med_income'] = (median_df['Med_Income'].astype(int) - avg) / stdev
median_df.head(5)

C:\Users\Jason\AppData\Local\Temp\ipykernel_24372\715520878.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  median_df['z_med_income'] = (median_df['Med_Income'].astype(int) - avg) / stdev


,Location,TimeFrame,DataFormat,Med_Income,CD,Med Income Str,z_med_income
0,Astoria,2023,Dollars,84590,401,"$84,590",0.057340
4,Battery Park/Tribeca,2023,Dollars,198945,101,"$198,945",3.079454
8,Bay Ridge,2023,Dollars,88566,310,"$88,566",0.162416
12,Bayside,2023,Dollars,107607,411,"$107,607",0.665621
16,Bedford Park,2023,Dollars,42387,207,"$42,387",-1.057978


In [27]:
median_df.to_csv('cleaned_data/median_income_df.csv', index=False)